# 01 · Baseline: regresión logística

Modelo de referencia rápido: regresión logística con un puñado de features crudas de la publicación (sin vendedor, sin texto, sin target encoding). Sirve para medir cuánto aporta todo lo demás.

In [1]:
import os, time, numpy as np, pandas as pd, warnings
import features as F
warnings.filterwarnings("ignore")
os.makedirs("models", exist_ok=True)
y = F.load_y(); folds = F.cv_folds(y)

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
X = pd.read_parquet(f"{F.CACHE}/Xtr.parquet"); X_test = pd.read_parquet(f"{F.CACHE}/Xte.parquet")

In [2]:
NUM = ["log_price", "initial_quantity", "sold_quantity", "available_quantity", "n_pictures", "warranty_null",
       "free_shipping", "local_pick_up", "automatic_relist", "accepts_mercadopago", "title_len"]
CAT = ["listing_type_id", "buying_mode", "status"]

def design(df):
    D = df[NUM].copy()
    for c in ["initial_quantity", "sold_quantity", "available_quantity"]:
        D[c] = np.log1p(D[c])
    return pd.concat([D, pd.get_dummies(df[CAT].astype(str), dtype=float)], axis=1)

D, D_test = design(X), design(X_test)
D_test = D_test.reindex(columns=D.columns, fill_value=0)
make = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

In [3]:
oof = np.zeros(len(y))
for f, (a, b) in enumerate(folds):
    oof[b] = make().fit(D.iloc[a], y[a]).predict_proba(D.iloc[b])[:, 1]
    print(f"fold {f}: accuracy {((oof[b] > .5) == y[b]).mean():.4f}")
print(f"\nAccuracy CV: {((oof > .5) == y).mean():.4f}   (clase mayoritaria: {max(y.mean(), 1 - y.mean()):.4f})")
np.save(f"{F.CACHE}/oof_baseline.npy", oof)

fold 0: accuracy 0.8261


fold 1: accuracy 0.8226


fold 2: accuracy 0.8220


fold 3: accuracy 0.8254


fold 4: accuracy 0.8212

Accuracy CV: 0.8234   (clase mayoritaria: 0.5370)


In [4]:
import joblib
model = make().fit(D, y)
joblib.dump(model, "models/baseline.joblib")
proba = model.predict_proba(D_test)[:, 1]
np.save(f"{F.CACHE}/test_baseline.npy", proba)
F.save_submission(proba, "baseline")["condition"].value_counts(normalize=True)

condition
used    0.500133
new     0.499867
Name: proportion, dtype: float64